# 长窗口数据交接复核

范围：2023-01-01至2026-09-30，香港时间。这里只复核真实下载、数据质量及暂定标签，不训练模型。原始记录、五分钟观测和拥堵事件的条数不能混用。

In [1]:
from pathlib import Path
import json
import pandas as pd

# 从 project/notebooks 运行；如将数据移到其他位置，请修改 DATA_ROOT。
DATA_ROOT = Path.cwd().parents[1]
assert (DATA_ROOT / "completion.json").exists(), "请先完成数据准备，并检查 DATA_ROOT"


## 官方档案是否完整

complete 必须为 true；45个观测月份与额外一天的边界档案分开统计。

In [2]:
manifest = json.loads((DATA_ROOT / "download_manifest.json").read_text(encoding="utf-8"))
print("完整:", manifest["complete"], "月份:", len(manifest["records"]))
print("原始ZIP字节数:", sum(r["bytes"] for r in manifest["records"]))
print(pd.DataFrame(manifest["records"])[["month", "bytes", "snapshots"]].to_string(index=False))


完整: True 月份: 45
原始ZIP字节数: 47070208218
 month      bytes  snapshots
202301 1136915260      41330
202302 1102041259      33451
202303 1177836381      36057
202304 1405115518      41348
202305 1456227624      42861
202306 1153123362      36124
202307 1387829813      42296
202308 1313636937      39242
202309 1244326576      38295
202310 1123921327      34547
202311  904144442      27224
202312 1137688330      34904
202401  839377136      25226
202402  833018465      25619
202403 1025498143      28462
202404  971931270      27519
202405 1095792123      32717
202406 1101642377      32207
202407 1156540800      32630
202408 1165212212      32646
202409 1124501732      31375
202410 1170653944      32516
202411 1113891644      30870
202412  990329259      27225
202501 1005493122      27308
202502  969126863      26563
202503 1238775278      34158
202504 1192919975      33115
202505 1212124927      33192
202506 1216974752      32780
202507  949617514      26100
202508 1039371057      28444
20250

## 站点选择和历史参考

只用2023年1月1日至7日筛选站点，1月拟合速度参考。未来停用和缺失不会导致事后换站。

In [3]:
cohort = pd.read_csv(DATA_ROOT / "reports/cohort.csv")
print(cohort.to_string(index=False))


detector_id  direction     lane_id  baseline_kmh  reference_bins       reference_fit_start         reference_fit_end
   AID01101 South East Middle Lane          64.5            6752 2023-01-01 00:00:00+08:00 2023-02-01 00:00:00+08:00
   AID01103 South East Middle Lane          68.5            6747 2023-01-01 00:00:00+08:00 2023-02-01 00:00:00+08:00
   AID01104 North East   Slow Lane          71.0            6613 2023-01-01 00:00:00+08:00 2023-02-01 00:00:00+08:00
   AID01105 North East   Slow Lane          73.0            6762 2023-01-01 00:00:00+08:00 2023-02-01 00:00:00+08:00
   AID01106 South East   Fast Lane          58.0            6707 2023-01-01 00:00:00+08:00 2023-02-01 00:00:00+08:00
   AID01107 North East   Fast Lane          71.0            6764 2023-01-01 00:00:00+08:00 2023-02-01 00:00:00+08:00
   AID01108 North West   Fast Lane          74.0            6617 2023-01-01 00:00:00+08:00 2023-02-01 00:00:00+08:00
   AID01111 North East   Fast Lane          69.0            6743

## 按月覆盖

30秒有效观测覆盖率和五分钟可用分箱比例是不同指标；空缺仍是空缺，不补成0。下面先显示最差的20个站点月份。

In [4]:
monthly = pd.read_csv(DATA_ROOT / "reports/coverage_detector_month.csv")
print(monthly.sort_values("usable_5m_coverage").head(20).to_string(index=False))


detector_id  direction   month  lane_days  valid_slots  expected_slots  usable_5m_bins  conflict_slots  offline_slots  valid_30s_coverage  usable_5m_coverage
   AID01111 North East 2026-07        124            0          357120               0               0              0            0.000000            0.000000
   AID01105 North East 2026-07         62            0          178560               0               0              0            0.000000            0.000000
   AID01105 North East 2026-06         60            0          172800               0               0              0            0.000000            0.000000
   AID01105 North East 2026-05         62            0          178560               0               0              0            0.000000            0.000000
   AID01111 North East 2026-06        120            0          345600               0               0              0            0.000000            0.000000
   AID01111 North East 2026-05        124           

## 标签数量和各时间分组

标签是预测时刻之后15/30/60分钟内是否观察到持续恢复确认。unknown不能当作未恢复。purged表示为防止事件跨时间边界而隔离的记录。

In [5]:
summary = json.loads((DATA_ROOT / "reports/summary.json").read_text(encoding="utf-8"))
print(json.dumps({"labels": summary["labels"], "splits": summary["splits"]}, ensure_ascii=False, indent=2))


{
  "labels": {
    "events": 7597,
    "horizons": {
      "15": {
        "positive": 1451,
        "negative": 5503,
        "unknown": 643,
        "reasons": {
          "observed_without_recovery": 5503,
          "confirmed_recovery": 1451,
          "missing_or_invalid_bin": 643
        }
      },
      "30": {
        "positive": 2968,
        "negative": 3831,
        "unknown": 798,
        "reasons": {
          "observed_without_recovery": 3831,
          "confirmed_recovery": 2968,
          "missing_or_invalid_bin": 798
        }
      },
      "60": {
        "positive": 4727,
        "negative": 1884,
        "unknown": 986,
        "reasons": {
          "confirmed_recovery": 4727,
          "observed_without_recovery": 1884,
          "missing_or_invalid_bin": 986
        }
      }
    }
  },
  "splits": {
    "test": {
      "events": 1529,
      "horizons": {
        "15": {
          "positive": 332,
          "negative": 1064,
          "unknown": 133,
          

## 最终检查与解释边界

检查通过不等于阈值已获得小组批准，也不证明事故因果关系或精确恢复时刻。长期设备/道路变化、稀疏观测与参考速度漂移仍须在实验中说明。

In [6]:
validation = json.loads((DATA_ROOT / "reports/validation.json").read_text(encoding="utf-8"))
print(json.dumps(validation, ensure_ascii=False, indent=2))
print("后续工作：道路匹配和空间特征；外部因素；基础模型、机器学习及时间评估。")


{
  "status": "passed",
  "checks": [
    "45 official monthly archives plus next-day boundary",
    "ZIP snapshot counts reconcile with processed capture entries including duplicate names",
    "All compact capture files and final handoff outputs match recorded SHA-256",
    "One unique event ID/key, binary-or-unknown labels, time-aware prediction dates",
    "One cleaned partition per observation day, valid domains, all conflicting/offline measurements masked"
  ],
  "cleaned_lane_observations": {
    "rows": 149984588,
    "valid_speed_rows": 147790583,
    "valid_occupancy_rows": 147790583,
    "valid_volume_rows": 147790583,
    "conflicts": 426,
    "unmasked_conflicts": 0,
    "unmasked_offline": 0,
    "invalid_domain_values": 0,
    "first_observation": "2023-01-01 00:00:30+08",
    "last_observation": "2026-10-01 00:00:00+08"
  },
  "event_rows": 7597,
  "limitations": "No claim of physical lane stability, incident causality, exact recovery time, model accuracy, or scientific